In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

In [4]:
df = sns.load_dataset("titanic")

In [5]:
# 1. features and label
data = df[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
data["is_female"] = (data["sex"] == "female").astype(int)
X = data[["pclass", "is_female", "age", "fare", "sibsp", "parch"]]
y = data["survived"]
print(X.shape, y.shape, round(y.mean(), 4))
print(X.isna().sum().to_dict())

(891, 6) (891,) 0.3838
{'pclass': 0, 'is_female': 0, 'age': 177, 'fare': 0, 'sibsp': 0, 'parch': 0}


In [7]:
# 2. split WITHOUT stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, round(y_train.mean(), 4), round(y_test.mean(), 4))

(712, 6) (179, 6) 0.3764 0.4134


In [8]:
# 3. split WITH stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape, round(y_train.mean(), 4), round(y_test.mean(), 4))

(712, 6) (179, 6) 0.3834 0.3855


In [10]:
# 4. what random_state does
a, _, _, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
b, _, _, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(a.index.equals(b.index)) #same seed

True


In [11]:
c, _, _, _ = train_test_split(X, y, test_size=0.2, stratify=y)
d, _, _, _ = train_test_split(X, y, test_size=0.2, stratify=y)
print(c.index.equals(d.index))          # no seed

False


In [12]:
# 5. the column you must NOT use as a feature
print(df[["survived", "alive"]].head(3).values.tolist())

[[0, 'no'], [1, 'yes'], [1, 'yes']]


1. Baseline. The dumbest model possible. DummyClassifier(strategy="most_frequent") always predicts the most common class. A real model only counts if it beats the baseline.

2. Leakage means information the model should not have gets into training. There are two kinds:

Target leakage: a feature that contains the answer (alive).

Preprocessing leakage: a number used for cleaning (median, mean, scaling) that was computed from ALL rows, including test rows. Rule: compute it from the training data only, then apply it to both parts.

3. Three-way split.

Train teaches the weights.

Validation compares settings and models.

Test gives the final score, once. If you keep tuning on the test set, the test set becomes part of training and the final score is too optimistic.

4. Pipeline. 
Pipeline([("imputer", ...), ("model", ...)]) chains the steps. .fit(X_train, y_train) fits every step on training data only. .score(X, y) then applies the same steps to whatever you give it. You cannot make the preprocessing mistake by accident.

Functions you use (each with its input and output):

DummyClassifier(strategy="most_frequent") builds the baseline model.

LogisticRegression(max_iter=1000) builds a model (a black box today).

SimpleImputer(strategy="median") fills NaN with the median of each training column.

.fit(X_train, y_train) learns from the training data.

.score(X, y) returns accuracy: the share of correct predictions.

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 1. baseline
base = DummyClassifier(strategy="most_frequent")
base.fit(X_train, y_train)
print("baseline", round(base.score(X_test, y_test), 4))

baseline 0.6145


In [19]:
# 2. the median must come from TRAIN only
print(X_train["age"].median(), X["age"].median(), X_test["age"].median())

28.5 28.0 27.0


In [20]:
# 3. the RIGHT way, by hand
m = X_train["age"].median()
X_train_f = X_train.copy()
X_test_f = X_test.copy()
X_train_f["age"] = X_train_f["age"].fillna(m)
X_test_f["age"] = X_test_f["age"].fillna(m)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_f, y_train)
print("right", round(lr.score(X_train_f, y_train), 4), round(lr.score(X_test_f, y_test), 4))

right 0.7949 0.8045


In [21]:
# 4. the WRONG way: fill with the median of ALL rows, then split
X_wrong = X.copy()
X_wrong["age"] = X_wrong["age"].fillna(X_wrong["age"].median())
a, b, c, d = train_test_split(X_wrong, y, test_size=0.2, random_state=42, stratify=y)
lw = LogisticRegression(max_iter=1000)
lw.fit(a, c)
print("wrong", round(lw.score(a, c), 4), round(lw.score(b, d), 4))

wrong 0.7921 0.8045


In [23]:
# 5. the Pipeline (X_train still contains NaN)
pipe= Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LogisticRegression(max_iter=1000))])
pipe.fit(X_train, y_train)
print("pipe", round(pipe.score(X_train, y_train), 4), round(pipe.score(X_test, y_test), 4))
print(pipe.named_steps["imputer"].statistics_.round(2))

pipe 0.7949 0.8045
[ 3.    0.   28.5  14.45  0.    0.  ]


In [25]:
# 6. validation set: split the TRAIN part again, test stays untouched
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
print(X_tr.shape, X_val.shape, X_test.shape)
for C in (0.01, 1.0):
    p = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LogisticRegression(C=C, max_iter=1000)),
    ])
    p.fit(X_tr, y_tr)
    print(C,round(p.score(X_tr, y_tr), 4), round(p.score(X_val, y_val), 4))

(534, 6) (178, 6) (179, 6)
0.01 0.736 0.7079
1.0 0.7996 0.7697


In [26]:
# 7. a real leak
X_leak = X.copy()
X_leak["alive_yes"] = (df["alive"] == "yes").astype(int)
a, b, c, d = train_test_split(X_leak, y, test_size=0.2, random_state=42, stratify=y)
pl = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000)),
])
pl.fit(a, c)
print("leak", pl.score(b, d))

leak 1.0
